# Sepang counterfactual: what if Hamilton had started on inters?

2026 Bahrain Grand Prix (held at Sepang, Round 16, 4 Oct 2026). Ferrari started Hamilton from P2 on slicks on a wet track. He finished P3.
This notebook runs the whole analysis end to end. Every number comes from `config.yaml` and the code in `src/`.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the notebook runs from notebooks/; src/ lives at the repo root
import fastf1; fastf1.set_log_level("ERROR")  # hide download chatter

from src.data_loader import fact_check, fix_timing, load_config, load_session

cfg = load_config()
session = load_session(cfg)

## M1: what does the data actually say?

Before modelling anything, check the race story against the timing data. The PRD's rule is that **the data wins**.

In [2]:
fact_check(session, cfg)


=== Event ===
Bahrain Grand Prix | Kuala Lumpur | 2026-10-04 | 55 laps

=== Official classification (top 6 + Ferraris) ===
                     TeamName  GridPosition  Position    Status  gap_s
Abbreviation                                                          
VER           Red Bull Racing           1.0       1.0  Finished  0.000
ANT                  Mercedes           3.0       2.0  Finished  2.307
HAM                   Ferrari           2.0       3.0  Finished  4.919
LEC                   Ferrari           4.0       4.0  Finished  7.258
HAD           Red Bull Racing           8.0       5.0  Finished  8.571
PIA                   McLaren           6.0       6.0  Finished  9.454

=== How the race started (race control) ===
 Lap                                                                                                                         Message
   1                                                                                                                   DELAYED STA

### What the data changed

| PRD / press said | Data says | Why it matters |
|---|---|---|
| Wet race | **No rain fell during the race.** It stopped about 54 min before the start, so the race ran on a drying track | Inter pace improves lap by lap, so a single "inter pace" number is wrong |
| Hamilton P2 → ~P16 on lap 1 | Laps 1–2 were **behind the SC**, then a **standing start**. HAM was P15 after lap 3 (the first racing lap) | The PRD's "lap-1 delta" becomes a lap-3 standing-start delta |
| Long wet phase, then a crossover | Wet phase = **laps 3–8 only**. All the leaders pitted for slicks on **lap 9, under the SC** | The crossover stop was cheap for everyone. The ghost gets the same cheap stop |
| Gamble cost Hamilton the race | He was **83 s** behind after lap 8, but only **6.5 s** after the SC, at P15 | The SC refunded the *time* but not the *track position*. Then he had to pass 10 cars |
| Leclerc P4 | Official P4, but FastF1 timing puts him P17 (+26.8 s) | A per-car timing glitch: +19.5 s on laps 48 and 51, under the late SC. M3 shifts his times back from lap 48. That puts him P5 at the restart and P4 at the flag, matching the official result |

**FastF1 lessons:**
- If the classification API is down, `session.results` quietly falls back to timing-derived positions. `load_session` refuses to continue if that happens.
- Unknown compounds come through as the *string* `'None'`.
- `TrackStatus` doesn't show the SC at the race start.
- Count a lap as SC/VSC only when most cars saw it, because back-markers' laps overlap the leaders' SC.

## M2: race phases and pace models

### Phases
Each lap gets a **track condition** (wet up to the crossover lap, dry after) and a **neutralisation** (green / SC / VSC).
Three independent signals put the crossover on lap 9:
- the leaders' pit stops;
- the slick-minus-inter lap-time gap reaching ~0;
- the SC itself.

In [3]:
from src.segmentation import phase_table
from src.pace_model import add_lap_features, fit_pace, inter_pace, start_gaps
from src.pit_loss import pit_stops, pit_loss_summary

phases = phase_table(session, cfg)
phases.loc[1:20]

,condition,neutral,crossover,slick_minus_inter_s,field_median_s
LapNumber,,,,,
1,wet,sc,False,NaN,NaN
2,wet,sc,False,NaN,NaN
3,wet,green,False,26.57,116.35
4,wet,green,False,21.02,113.00
5,wet,green,False,14.25,112.85
6,wet,green,False,10.33,113.32
7,wet,green,False,4.71,114.21
8,wet,green,False,0.48,113.93
9,wet,sc,True,NaN,NaN


`field_median_s` keeps falling until lap 16, so **the track was still drying after the crossover**.
That's why the dry fit starts at lap 17 (`pace.dry_fit_start_lap`). Before that, track drying would be mistaken for tyre wear.

### Clean laps
A lap is usable for pace only if it is:
- not an in-lap or out-lap;
- not behind the SC/VSC, and not lap 1–3;
- at least `pace.traffic_gap_s` behind the car ahead at the line (lapped or not);
- not flagged inaccurate by FastF1, and not deleted for track limits.

LEC's laps use the corrected timing (`fix_timing`, see M3), so his gap to the car ahead is right after lap 48.

**Effective tyre age** counts only green dry laps as wear (`pace.wet_sc_wear` = 0). So HAM's softs, used in the wet from lap 1, only start ageing at lap 13.

In [4]:
df = add_lap_features(fix_timing(session, cfg), phases, cfg)
print(f"clean laps: {int(df.clean.sum())} of {len(df)}")
df[df.Driver == "HAM"].set_index("LapNumber")[["lap_s", "Compound", "TyreLife", "eff_age", "gap_ahead", "clean"]].loc[[8, 13, 17, 25, 30, 33, 52, 55]].round(2)

clean laps: 561 of 1145


,lap_s,Compound,TyreLife,eff_age,gap_ahead,clean
LapNumber,,,,,,
8,112.88,SOFT,8.0,0.0,4.39,True
13,110.98,SOFT,13.0,1.0,0.49,False
17,102.78,SOFT,17.0,5.0,0.46,False
25,101.73,SOFT,25.0,13.0,10.14,True
30,102.50,SOFT,30.0,18.0,5.09,True
33,100.18,MEDIUM,5.0,5.0,0.83,False
52,99.62,SOFT,12.0,4.0,0.29,False
55,99.44,SOFT,15.0,7.0,2.60,True


### Dry pace: one fit over the whole field

`lap_s = <compound> + deg_<compound> * eff_age + remaining * laps_remaining + d_<driver>`, with HAM as the reference (d = 0).

**Why the whole field?** A HAM + LEC fit gave deg 0.19 s/lap and LEC 1.1 s slower. Comparing HAM's laps 17–30 with 52–55 mixes three things: tyre wear, fuel burn and the track rubbering in.
On any single lap, 20 cars run very different tyre ages. That's what separates deg from `remaining` (fuel + track, *measured* at ~0.044 s/lap).
`d_<drv>` is each driver's offset to HAM (positive = slower).

In [5]:
fit = fit_pace(df, cfg)
ip = inter_pace(df, cfg)
table = pd.DataFrame({"coef": fit["coef"], "se": np.sqrt(np.diag(fit["cov"]))}).round(3)
table[~table.index.str.startswith("d_") | table.index.isin([f"d_{d}" for d in ["LEC"] + ip["leaders"]])]

,coef,se
HARD,98.956,0.296
MEDIUM,99.535,0.335
SOFT,98.824,0.233
UNKNOWN,98.418,1.007
deg_HARD,0.184,0.013
deg_MEDIUM,0.111,0.021
deg_SOFT,0.147,0.015
deg_UNKNOWN,0.105,0.074
remaining,0.043,0.004
d_ANT,-0.265,0.236


### Inter pace for HAM, who never ran inters

HAM's inter pace is modelled as an offset (s/lap) to `ref`, the leaders' median clean inter lap on each wet lap. `ref` follows the drying track.
- **Proxy**: LEC's inter laps minus `ref`, then adjusted by the HAM−LEC dry gap.
- **Ratio**: HAM's dry gap to the leaders, scaled by the wet/dry lap-length ratio.

The two disagree by about 1.4 s/lap. The proxy is pulled faster for two reasons:
- LEC's inters were **3 laps fresher** than the leaders' on a drying track;
- LEC was **~1 s/lap slower in the dry** (robust to the traffic filter; possibly damage from his lap-3 collision), and that gap gets added back.

So the Monte Carlo samples Normal(mu, sigma), with sigma wide enough to cover both. `pace.proxy_weight` sets mu. The M5 tornado tests it.

In [6]:
print("reference:", ip["ref"].round(2).to_dict())
print(f"proxy {ip['proxy']:+.2f} +/- {ip['proxy_se']:.2f} | ratio {ip['ratio']:+.2f} +/- {ip['ratio_se']:.2f} "
      f"| Normal(mu={ip['mu']:+.2f}, sigma={ip['sigma']:.2f}) s/lap")
start_gaps(df, session.results, cfg).round(2)

reference: {4: 109.08, 5: 110.46, 6: 111.37, 7: 112.65, 8: 112.62}
proxy -1.64 +/- 0.42 | ratio -0.23 +/- 0.20 | Normal(mu=-0.94, sigma=0.78) s/lap


,grid,pos,gap_s
Driver,,,
VER,1.0,3.0,2.26
ANT,3.0,1.0,0.00
RUS,7.0,2.0,0.87
HAD,8.0,5.0,5.14


### Pit loss

Loss = (in-lap − ref) + (out-lap − ref).
- **Green stops:** the reference is the driver's own clean pace nearby. The median comes out at about 26.5 s.
- **Neutralised stops:** the reference is the median lap of the cars that didn't pit.

That reference **breaks when the SC comes out mid-lap** (laps 9 and 43). Cars that had already done most of the lap at racing speed look cheap (VER ~10 s); cars further back look expensive (~24 s). The loss depends on *where you were* when the SC came out.
So M3 won't use a single SC number. For a stop on a neutralised lap, the ghost interpolates between the cars that stopped on that same lap just ahead of it and just behind it on the road.

In [7]:
stops = pit_stops(df, cfg)
pit_loss_summary(stops)

,n,median_s,min_s,max_s
neutral,,,,
green,28,26.54,23.38,52.05
sc,31,18.29,4.95,24.68
vsc,7,21.28,1.11,46.20


## M3: the simulator and the back-test

`src/simulator.py` races one **ghost car** against the other 19 cars' *recorded* lap times. The real cars don't react to it. On each lap:

| Lap type | What the ghost does | Where the number comes from |
|---|---|---|
| Green | Lap time = dry fit + that lap's **lap effect** | M2 fit; median residual of the field that lap |
| Green, catching cars | Each car it catches (in road order) costs the **pass cost**. If it's still ahead after paying, it's a pass; if not, it's stuck behind that car | Measured from the other drivers (not HAM): 0.66 ± 0.25 s |
| Green pit stop | + green pit loss on the in-lap | Median of the others' green stops |
| SC | Follows the nearest car ahead. Its gap shrinks to at most the field's median gap at the line that lap | The queue forming: 4.5 → 3.6 → 1.4 → 0.6 s over laps 9–12 |
| VSC | Gap to the car ahead frozen | Everyone slows by the same delta |
| Stop under SC/VSC | Interpolates the real stoppers just ahead of and just behind it on the road | What a stop costs depends on where you were when the SC came out |

**Lap effects** show the track still drying after the crossover: every car was 6.1 / 3.1 / 1.9 / 0.6 s slower than the fit on laps 13–16. From lap 17 they're within ±0.3 s.

The engine is generic: any driver, race or tyre plan. `src/pipeline.py` feeds it this race.

In [8]:
from src.pipeline import prepare, backtest

p = prepare(cfg)
print("pass cost (s per car passed, HAM excluded):", {k: round(v, 2) for k, v in p["pass_cost"].items()})
print("lap effects, laps 13-20:", p["effects"].loc[13:20].round(2).to_dict())

pass cost (s per car passed, HAM excluded): {'mean': 0.66, 'se': 0.25, 'n_passes': 49}
lap effects, laps 13-20: {13: 6.14, 14: 3.13, 15: 1.94, 16: 0.62, 17: 0.07, 18: -0.09, 19: -0.1, 20: -0.08}


### Back-test: replay HAM's real strategy

The ghost takes over from HAM's **real** state at the end of lap 8 (`backtest.start_lap`). Laps 3–8 were slicks on a wet track, and nothing in the model predicts that (D-005).
From there it runs his real plan: softs → medium (green stop, lap 31) → softs (VSC stop, lap 43). It uses mean pace and no noise.

**Pass criteria** (`backtest.tolerance_s` = 5 s):
- the **flag**: P3 and within 5 s;
- **lap 42**, before the late SC: within 5 s. The late SC resets every gap, so the flag alone would hide errors (D-006).

In [9]:
bt = backtest(p, cfg)
bt.loc[[12, 13, 16, 19, 22, 30, 32, 42, 44, 51, 52, 55],
       ["position", "real_position", "passes", "lap_s", "real_lap_s", "error_s"]].round(2)

,position,real_position,passes,lap_s,real_lap_s,error_s
LapNumber,,,,,,
12,15,15.0,0,150.92,128.06,-0.19
13,11,12.0,4,110.23,110.98,-0.93
16,9,9.0,0,102.38,102.04,-0.51
19,7,7.0,1,102.63,102.50,-0.97
22,5,5.0,0,101.73,101.86,-0.88
30,5,5.0,0,102.53,102.50,-1.21
32,7,7.0,0,101.02,120.48,-0.31
42,5,5.0,0,102.14,101.68,0.70
44,5,6.0,0,151.35,NaN,-2.78


**What the back-test caught.** The first run had free passing (the PRD default). It finished P3 within 0.7 s, but it was **11.9 s ahead at lap 42**:
- the error built up entirely on laps 13–21, while the real HAM carved from P15 to P5;
- it was flat in clean air.

So the late SC had been hiding a traffic error, which is exactly why lap 42 is checked.

The fix was **not** to set the penalty to whatever makes HAM's race fit (about 1.2 s per pass). That would calibrate the model on the test. Instead:
1. **Measure** the cost per pass from the other 19 drivers: 0.66 ± 0.25 s.
2. Make the rule realistic: each car caught costs the penalty, and if the ghost can't get by, it's stuck behind.

With both, the replay tracks HAM's climb within about 1 s and 1–2 places.

**Weak spots:**
- **This is an in-sample check.** HAM's clean laps are in the pace fit; only the pass cost is held out.
- **Mid-SC positions are rough.** A car far back closes up in one lap, so only the restart order is reliable.
- **The VSC stop on lap 43 comes out 3–4 s cheaper** than HAM's real stop. The interpolated stoppers were 29–34 s away on the road.
- **The green pit loss sits entirely on the in-lap.** That's why lap 31 shows a +19 s blip that is gone by lap 32.